## Project: Geospatial Analysis using Google Places API

### Authors:
1 - Angel Francisco Cruz Brito  
2 - Pablo Enrique Santini Buenfil

### Libraries

In [53]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point, box
import folium
import time
import requests
import pandas as pd
from dotenv import load_dotenv
import os

### 1 - Generating the Spatial Grid

To begin the analysis, we first need to identify all the parks within the Merida metropolitan area. Since the Google Places API limits results to a specific radius from a central point, we will use the GeoPandas library to automatically generate a spatial grid. This will allow us to iterate over the map and ensure total coverage of the study area.

In [8]:
city_bbox = [-89.74489050365537, 20.880264095990857, -89.51284567363703, 21.08566549700958]
radio_busqueda = 1700 

In [3]:
def generar_centros_hexagonales(bbox_latlon, radio_metros, crs_utm="EPSG:32616"):
    bbox_geom = box(*bbox_latlon)
    gdf_bbox = gpd.GeoDataFrame({'geometry': [bbox_geom]}, crs="EPSG:4326")
    gdf_bbox_utm = gdf_bbox.to_crs(crs_utm)
    minx, miny, maxx, maxy = gdf_bbox_utm.total_bounds
    
    dx = radio_metros * np.sqrt(3)
    dy = radio_metros * 1.5
    centros_utm = []
    
    row = 0
    y = miny
    while y <= maxy + dy:
        x_offset = (dx / 2.0) if (row % 2 != 0) else 0.0
        x = minx + x_offset
        while x <= maxx + dx:
            centros_utm.append(Point(x, y))
            x += dx
        y += dy
        row += 1
        
    gdf_puntos_utm = gpd.GeoDataFrame(geometry=centros_utm, crs=crs_utm)
    gdf_puntos_recortados = gpd.clip(gdf_puntos_utm, gdf_bbox_utm)
    
    return gdf_puntos_recortados.to_crs("EPSG:4326")

In [4]:
puntos_api = generar_centros_hexagonales(city_bbox, radio_busqueda)

In [5]:
centro_lat = (city_bbox[1] + city_bbox[3]) / 2
centro_lon = (city_bbox[0] + city_bbox[2]) / 2
map = folium.Map(location=[centro_lat, centro_lon], zoom_start=11)

folium.Rectangle(
    bounds=[[city_bbox[1], city_bbox[0]], [city_bbox[3], city_bbox[2]]],
    color="red", fill=False, weight=2, tooltip="Zona Límite"
).add_to(map)

for index, row in puntos_api.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x
    
    folium.Circle(
        location=[lat, lon],
        radius=radio_busqueda, 
        color='blue',
        weight=1,
        fill=True,
        fill_color='blue',
        fill_opacity=0.15,
        tooltip=f"Lat: {lat:.4f}, Lon: {lon:.4f}"
    ).add_to(map)
    
map

In [86]:
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")
radius = 1700
search_type = ["park"]
parques = []

In [87]:
for index, row in puntos_api.iterrows():
    lat = float(row.geometry.y)
    lon = float(row.geometry.x)
    
    print(f"Escaneando nodo {index + 1}/{len(puntos_api)} (Lat: {lat:.4f}, Lon: {lon:.4f})...")
    
    url = "https://places.googleapis.com/v1/places:searchNearby"
    
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": "places.id,places.displayName,places.location,places.rating,places.userRatingCount,places.formattedAddress,places.types"
    }
    
    payload = {
        "includedTypes": search_type,
        "maxResultCount": 20, 
        "locationRestriction": {
            "circle": {
                "center": {
                    "latitude": lat,
                    "longitude": lon
                },
                "radius": radius
            }
        }
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        data = response.json()
        
        if "error" in data:
            print(f"Error de la API en el nodo {index}: {data['error']['message']}")
            continue 
        
        places = data.get("places", [])
        
        for place in places:
            park_info = {
                "place_id": place.get("id"),
                "name": place.get("displayName", {}).get("text"),
                "lat": place.get("location", {}).get("latitude"),
                "lon": place.get("location", {}).get("longitude"),
                "rating": place.get("rating"),
                "user_ratings_total": place.get("userRatingCount", 0),
                "address": place.get("formattedAddress"),
                "types": ",".join(place.get("types", []))
            }
            parques.append(park_info)
            
    except Exception as e:
        print(f"Error de conexión en el nodo {index}: {e}")
        continue

    time.sleep(2)

Escaneando nodo 11/68 (Lat: 20.8999, Lon: -89.7311)...
Escaneando nodo 12/68 (Lat: 20.9004, Lon: -89.7028)...
Escaneando nodo 21/68 (Lat: 20.9232, Lon: -89.7174)...
Escaneando nodo 22/68 (Lat: 20.9236, Lon: -89.6891)...
Escaneando nodo 30/68 (Lat: 20.9460, Lon: -89.7319)...
Escaneando nodo 31/68 (Lat: 20.9464, Lon: -89.7036)...
Escaneando nodo 13/68 (Lat: 20.9008, Lon: -89.6745)...
Escaneando nodo 14/68 (Lat: 20.9013, Lon: -89.6463)...
Escaneando nodo 15/68 (Lat: 20.9017, Lon: -89.6180)...
Escaneando nodo 23/68 (Lat: 20.9241, Lon: -89.6608)...
Escaneando nodo 24/68 (Lat: 20.9245, Lon: -89.6325)...
Escaneando nodo 32/68 (Lat: 20.9469, Lon: -89.6754)...
Escaneando nodo 33/68 (Lat: 20.9473, Lon: -89.6471)...
Escaneando nodo 34/68 (Lat: 20.9477, Lon: -89.6188)...
Escaneando nodo 42/68 (Lat: 20.9701, Lon: -89.6616)...
Escaneando nodo 43/68 (Lat: 20.9705, Lon: -89.6333)...
Escaneando nodo 51/68 (Lat: 20.9929, Lon: -89.6762)...
Escaneando nodo 52/68 (Lat: 20.9933, Lon: -89.6479)...
Escaneando

In [6]:
df_raw = pd.DataFrame(parques)

NameError: name 'parques' is not defined

In [110]:
if not df_raw.empty:
    df_raw = df_raw.drop_duplicates(subset=["place_id"])
    print(f"\nTotal de parques únicos encontrados: {len(df_raw)}")
    df_raw.to_csv("data/merida_parks.csv", index=False)
else:
    print("\nNo se encontraron parques")


Total de parques únicos encontrados: 839


In [7]:
df = pd.read_csv("data/merida_parks.csv")

In [8]:
df.head()

,place_id,name,lat,lon,rating,user_ratings_total,address,types
0,ChIJC9c7Al0TVo8RMOV8TK2KJkE,Uman Yucatán,20.890890,-89.732996,4.6,122,"San Lorenzo, Umán, 97390 Uman, Yucatan, Mexico","park,point_of_interest,establishment"
1,ChIJsaescYgSVo8Rf35zX2Lta8w,BOSQUES DE SAN FRANCISCO Park,20.890300,-89.736345,4.0,125,"Por 22A y 22C, Calle 15-B, Bosques De San Fran...","park,point_of_interest,establishment"
2,ChIJxXVBDF4TVo8Ruu3jv6y2YOU,brisas de Uman Park,20.888373,-89.735986,3.9,23,"C. 6ꞌ 84, Umán, 97390 Umán, Yuc., Mexico","state_park,park,point_of_interest,establishment"
3,ChIJHTDxWt8TVo8R_M6v9cmADIc,ENTRADA SAN LORENZO Park,20.894012,-89.732722,NaN,0,"San Lorenzo, Umán, 97390 Umán, Yuc., Mexico","park,point_of_interest,establishment"
4,ChIJ1WAk4QATVo8RF-7sqXH_L6Q,tortugas Park,20.892380,-89.733638,NaN,0,"San Lorenzo, Umán, 97390 Umán, Yuc., Mexico","park,point_of_interest,establishment"


In [9]:
filtro_nombre = df['name'].str.contains(r'parque|park', case=False, na=False)


In [10]:
filtro_geografico = df['address'].str.contains(
    r'um[aá]n|conkal|kanas[ií]n|97357|9739\d|9737\d', 
    case=False, na=False
)

In [11]:
filtro_glorieta = df['name'].str.contains('glorieta', case=False, na=False)

In [12]:
df_merida = df.loc[
    filtro_nombre & 
    (filtro_geografico == False) & 
    (filtro_glorieta == False)
].copy()

In [13]:
df_merida

,place_id,name,lat,lon,rating,user_ratings_total,address,types
31,ChIJvwtVaWcNVo8RTyCVstEL5Mg,Paseos de Merida Park,20.933434,-89.707752,4.6,223,"Paseos De Mérida, 97312 Mérida, Yuc., Mexico","park,point_of_interest,establishment"
38,ChIJ41DfrLtyVo8RdibuQ8ZzhGk,BICENTENARIO Park,20.918154,-89.688615,4.3,142,"Bicentenario, 97255 Mérida, Yuc., Mexico","park,point_of_interest,establishment"
39,ChIJs0rMKZhyVo8RUUj2SA0s8b0,ROBLE AGRICOLA Park,20.914281,-89.679745,4.3,239,"Por calle 50, C. 30, El Roble Agrícola, 97255 ...","park,point_of_interest,establishment"
40,ChIJt3OduM9yVo8RigrHnz15oMI,Diamante Paseos de Opichen Park,20.936982,-89.692985,4.3,235,"entre calle 146 y calle 148, Perif. de Mérida ...","park,point_of_interest,establishment"
41,ChIJFQujhJZyVo8RtI67FqTkdns,El Roble Agricola Park,20.919945,-89.678371,4.4,165,"Esquina con 22, C. 45, El Roble, 97295 Mérida,...","park,point_of_interest,establishment"
...,...,...,...,...,...,...,...,...
822,ChIJl4t5hB95Vo8R6I8QpBGza5c,"Sitpach. ""K'íiwik Sitpach"" Park",21.027193,-89.520846,4.6,80,"97306 Sitpach, Yuc., Mexico","park,point_of_interest,establishment"
824,ChIJMQUXLQB5Vo8RxUHtE1Ftz4g,sitpach Main Park,21.026943,-89.521302,NaN,0,"Parque de, 97306 Sitpach, Yuc., Mexico","park,point_of_interest,establishment"
825,ChIJOSnlNwB5Vo8RSuhe8RZeYLI,Dunas Park,21.025614,-89.535278,4.5,2,"2FG7+6V, 97305 Cholul, Yucatan, Mexico","park,point_of_interest,establishment"
826,ChIJb_nICwB5Vo8Rmf7q-LZIhp0,Fiora Park,21.043667,-89.528918,NaN,0,"2FVC+4F, 97305 Merida, Yucatan, Mexico","park,point_of_interest,establishment"


In [14]:
centro_lat = (city_bbox[1] + city_bbox[3]) / 2
centro_lon = (city_bbox[0] + city_bbox[2]) / 2
map = folium.Map(location=[centro_lat, centro_lon], zoom_start=11)

folium.Rectangle(
    bounds=[[city_bbox[1], city_bbox[0]], [city_bbox[3], city_bbox[2]]],
    color="red", fill=False, weight=2, tooltip="Zona Límite"
).add_to(map)

for index, row in df_merida.iterrows():
    folium.Marker(
        location=[row['lat'],row['lon']],
        popup=f"<b>{row['name']}</b><br>Address: {row['address']}<br>Rating: {row['rating']}",
        tooltip=row['name'],
        icon=folium.Icon(color='green', icon='info-sign')
    ).add_to(map)
    
map

In [15]:
df_merida.to_csv("data/merida_parks_filtered.csv", index=False)

## 2 - Data Extraction
Later we cleaned data from first iteration, we read the dataframe generated. For visualization we display the result data on the map. 

In [54]:
final_df_merida = pd.read_csv("data/merida_parks_filtered.csv")

In [55]:
final_df_merida.head()

,place_id,name,lat,lon,rating,user_ratings_total,address,types
0,ChIJvwtVaWcNVo8RTyCVstEL5Mg,Paseos de Merida Park,20.933434,-89.707752,4.6,223,"Paseos De Mérida, 97312 Mérida, Yuc., Mexico","park,point_of_interest,establishment"
1,ChIJ41DfrLtyVo8RdibuQ8ZzhGk,BICENTENARIO Park,20.918154,-89.688615,4.3,142,"Bicentenario, 97255 Mérida, Yuc., Mexico","park,point_of_interest,establishment"
2,ChIJs0rMKZhyVo8RUUj2SA0s8b0,ROBLE AGRICOLA Park,20.914281,-89.679745,4.3,239,"Por calle 50, C. 30, El Roble Agrícola, 97255 ...","park,point_of_interest,establishment"
3,ChIJt3OduM9yVo8RigrHnz15oMI,Diamante Paseos de Opichen Park,20.936982,-89.692985,4.3,235,"entre calle 146 y calle 148, Perif. de Mérida ...","park,point_of_interest,establishment"
4,ChIJFQujhJZyVo8RtI67FqTkdns,El Roble Agricola Park,20.919945,-89.678371,4.4,165,"Esquina con 22, C. 45, El Roble, 97295 Mérida,...","park,point_of_interest,establishment"


In [5]:
def display_marker_info(row):
    return f'''
        <div style="min-width:200px; padding:5px;">
            <b>{row['name']}</b><br>
            Address: {row['address']}<br>
            Rating: {row['rating']}
        </div>
    '''

In [6]:
def display_parks_on_map(df, city_bbox):
    centro_lat = (city_bbox[1] + city_bbox[3]) / 2
    centro_lon = (city_bbox[0] + city_bbox[2]) / 2
    map = folium.Map(location=[centro_lat, centro_lon], zoom_start=11)

    folium.Rectangle(
        bounds=[[city_bbox[1], city_bbox[0]], [city_bbox[3], city_bbox[2]]],
        color="red", fill=False, weight=2, tooltip="Zona Límite"
    ).add_to(map)

    for index, row in df.iterrows():
        folium.Marker(
            location=[row['lat'],row['lon']],
            popup=display_marker_info(row),
            tooltip=row['name'],
            icon=folium.Icon(color='green', icon='info-sign')
        ).add_to(map)
    
    return map

In [9]:
map = display_parks_on_map(final_df_merida, city_bbox)
map

## 2.1 - Dataframe Formatting
We can extract more usable data in our dataframe we had to assign score for park relevance and extract the postal code for the next analysis

- **Postal Code Field:** Mérida's Postal Code of each city region, has five digits and always begins with `97`, we can use an regex filter for mask the dataframe for this field, so we applied to `address` field.

- **Type Attributes Field:** Some interest type as `sports`, we assign an mask to match the word, due to types appear as a string with comma separated values.

In [10]:
import re

In [11]:
# For get all types possible of the field "types" from the dataframe
types_values = final_df_merida['types'].str.split(',').explode().unique()
types_values

array(['park', 'point_of_interest', 'establishment', 'playground',
       'state_park', 'tourist_attraction', 'amusement_center',
       'amusement_park', 'city_park', 'athletic_field',
       'sports_activity_location', 'zoo', 'cell_phone_store', 'store',
       'nature_preserve', 'community_center', 'event_venue', 'service',
       'dog_park'], dtype=object)

In [12]:
# Filters
# Address Code -> Regex
address_code = final_df_merida['address'].apply(lambda x: re.search(r'\b97\d{3}\b', x).group(0)).astype(int)

# Interest Flags
has_sports = final_df_merida['types'].str.contains(
    'sports', 
    case=False, na=False
).astype(int)
has_playground = final_df_merida['types'].str.contains(
    'playground', 
    case=False, na=False
).astype(int)
allow_pets = final_df_merida['types'].str.contains(
    'dog_park', 
    case=False, na=False
).astype(int)

# Scoring
normalized_rating = final_df_merida['rating'].fillna(0) / final_df_merida['rating'].max()
normalized_user_ratings = final_df_merida['user_ratings_total'].fillna(0) / final_df_merida['user_ratings_total'].max()
w_rate_value = 0.3 
w_rate_count = 0.7 # Priorize user ratings count

#Assign fields to the dataframe
final_df_merida['address_code'] = address_code
final_df_merida['has_sports'] = has_sports
final_df_merida['has_playground'] = has_playground
final_df_merida['allow_pets'] = allow_pets
final_df_merida['score'] = (
    w_rate_value * normalized_rating + 
    w_rate_count * normalized_user_ratings
)

# Organize columns of the dataframe
final_df_merida = (
    final_df_merida.sort_values(by='score', ascending=False)
    .reset_index(drop=True)
)

final_df_merida.head()



,place_id,name,lat,lon,rating,user_ratings_total,address,types,address_code,has_sports,has_playground,allow_pets,score
0,ChIJRb2bT8FzVo8RNWkFMpW16qA,Parque Zoológico del Centenario,20.969283,-89.640160,4.6,30127,"Av. Itzaes s/n x 59, Parque Santiago, Centro, ...","zoo,tourist_attraction,park,point_of_interest,...",97000,0,0,0,0.976000
1,ChIJzxvb5vpzVo8RaI5jFziwIlY,Parque de las Américas,20.987509,-89.633218,4.7,19101,"Av. Colón 19-Por calle 20 y 23, García Ginerés...","park,point_of_interest,establishment",97070,0,0,0,0.725811
2,ChIJSTDOAthzVo8RFoUoZF9xios,Parque de San Juan,20.962713,-89.626177,4.4,18015,"C. 67A 529, Centro, 97000 Mérida, Yuc., Mexico","park,tourist_attraction,point_of_interest,esta...",97000,0,0,0,0.682578
3,ChIJySogAmFxVo8Rxkkp6OGhQgM,Parque de Santa Lucía,20.971199,-89.622596,4.7,14410,"C. 60 476A, Parque Santa Lucia, Centro, 97000 ...","park,tourist_attraction,point_of_interest,esta...",97000,0,0,0,0.616816
4,ChIJb0nabV5xVo8RWRDQlUNmCDM,Parque de Santa Ana,20.975855,-89.621170,4.6,12873,"C. 60 y 45, Parque Santa Ana, Centro, 97000 Mé...","park,tourist_attraction,point_of_interest,esta...",97000,0,0,0,0.575104


Now we visualize the obtained data via heatmap

In [13]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from folium.plugins import HeatMap 

In [14]:
def display_marker_score_info(row):
    return f'''
        <div style="min-width:200px; padding:5px;">
            <b>{row['name']}</b><br>
            Score: {row['score']:.2f}
        </div>
    '''

In [22]:
def display_parks_on_map(df, city_bbox, attr, color_map):
    centro_lat = (city_bbox[1] + city_bbox[3]) / 2
    centro_lon = (city_bbox[0] + city_bbox[2]) / 2
    map = folium.Map(location=[centro_lat, centro_lon], zoom_start=11)

    colormap = plt.get_cmap(color_map)

    for index, row in df.iterrows():
        color = mcolors.to_hex(colormap(row[attr]))
        folium.CircleMarker(
            location=[row['lat'],row['lon']],
            popup=display_marker_score_info(row),
            tooltip=row['name'],
            icon=folium.Icon(color='green', icon='info-sign'),
            fill=True,
            radius=5,
            color=color,
            fill_color=color,
            fill_opacity=0.7
        ).add_to(map)
    
    return map

In [23]:
score_map = display_parks_on_map(final_df_merida, city_bbox, 'score', 'Blues')
heatmap_data = final_df_merida[['lat', 'lon', 'score']].values.tolist()
score_map.add_child(HeatMap(heatmap_data, radius=30, blur=25, max_zoom=11))
score_map

We can observe, that we have more information about the popularity/quality on each park placed in the center of the city.

## 2.2 - Spatial Analysis & Enrichment (INEGI Data):

We downloaded the official 2020 demographic data and urban polygons (AGEBs) from INEGI. Using GeoPandas, we performed a Spatial Join to merge our map of parks with the population census. This allowed us to calculate exactly how many parks exist in each specific neighborhood.

In [56]:
gdf_ageb = gpd.read_file("data/31a.shp")
gdf_ageb['CVEGEO'] = gdf_ageb['CVEGEO'].str.strip()

df_censo = pd.read_csv("data/RESAGEBURB_31CSV20.csv", encoding='utf-8', low_memory=False)

df_censo_ageb = df_censo[df_censo['NOM_LOC'].str.contains('AGEB urbana', case=False, na=False)].copy()

df_censo_ageb['CVEGEO'] = (
    df_censo_ageb['ENTIDAD'].astype(float).astype(int).astype(str).str.zfill(2) + 
    df_censo_ageb['MUN'].astype(float).astype(int).astype(str).str.zfill(3) + 
    df_censo_ageb['LOC'].astype(float).astype(int).astype(str).str.zfill(4) + 
    df_censo_ageb['AGEB'].astype(str).str.strip().str.zfill(4)
)


gdf_ageb = gdf_ageb.merge(df_censo_ageb[['CVEGEO', 'POBTOT']], on='CVEGEO', how='inner')


gdf_ageb['POBTOT'] = pd.to_numeric(gdf_ageb['POBTOT'], errors='coerce').fillna(0)
gdf_ageb = gdf_ageb[gdf_ageb['POBTOT'] > 0]

In [57]:
gdf_parques = gpd.GeoDataFrame(
    final_df_merida, 
    geometry=gpd.points_from_xy(final_df_merida['lon'], final_df_merida['lat']),
    crs="EPSG:4326"
)

In [58]:
gdf_ageb = gdf_ageb.to_crs(gdf_parques.crs)

In [59]:
parques_con_ageb = gpd.sjoin(gdf_parques, gdf_ageb, how="inner", predicate="within")

In [60]:
conteo_parques = parques_con_ageb.groupby('CVEGEO').size().reset_index(name='cantidad_parques')

In [61]:
gdf_analisis = gdf_ageb.merge(conteo_parques, on='CVEGEO', how='left')

In [62]:
gdf_analisis['cantidad_parques'] = gdf_analisis['cantidad_parques'].fillna(0)

In [63]:
gdf_analisis['parques_por_1000_hab'] = (gdf_analisis['cantidad_parques'] / gdf_analisis['POBTOT']) * 1000

In [64]:
desiertos_criticos = desiertos_criticos[desiertos_criticos['CVEGEO'].str.startswith('31050')]

top_urgentes = desiertos_criticos[['CVEGEO', 'POBTOT', 'cantidad_parques']].sort_values(by='POBTOT', ascending=False)
print("Top 10 Mérida:\n", top_urgentes.head(10))

desiertos_criticos.to_file("data/desiertos_parques_prioritarios.geojson", driver='GeoJSON')

Top 10 Mérida:
             CVEGEO  POBTOT  cantidad_parques
765  3105000016045    5529               0.0
740  310500001295A    4103               0.0
898  3105001114405    4095               0.0
473  310500001053A    4090               0.0
770  3105000016721    3951               0.0
438  3105000013784    3838               0.0
599  3105000012786    3432               0.0
440  3105000010760    3389               0.0
516  3105000015583    3298               0.0
645  310500001288A    3279               0.0


In [65]:
city_bbox = [-89.74489050365537, 20.880264095990857, -89.51284567363703, 21.08566549700958]
centro_lat = (city_bbox[1] + city_bbox[3]) / 2
centro_lon = (city_bbox[0] + city_bbox[2]) / 2
mapa_merida = folium.Map(location=[centro_lat, centro_lon], zoom_start=12, tiles='CartoDB positron')

folium.GeoJson(
    desiertos_criticos,
    name="Zonas Críticas",
    style_function=lambda feature: {
        'fillColor': '#e63946',   
        'color': '#800000',       
        'weight': 2,
        'fillOpacity': 0.6
    },
    tooltip=folium.GeoJsonTooltip(
        fields=['CVEGEO', 'POBTOT'],
        aliases=['Clave AGEB:', 'Población sin parques:'],
        style=("background-color: white; color: #333333; font-family: arial; font-size: 12px; padding: 10px;")
    )
).add_to(mapa_merida)

for idx, row in gdf_parques.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color='#2a9d8f', 
        fill=True,
        fill_color='#2a9d8f',
        fill_opacity=0.8,
        tooltip=row['name']
    ).add_to(mapa_merida)

mapa_merida

## Conclusions
This observation suggests that the city of Mérida may need to improve the distribution of its parks. Whilst in certain areas the density of parks is high, it is evident that in a large number of blocks there are no parks at all (see Caucel).